# Arabic News Assistant - Finetuning Pipeline

This notebook provides a complete pipeline for finetuning Qwen models on Arabic news data.

## Steps:
1. Setup and Configuration
2. Data Generation (Knowledge Distillation)
3. Dataset Preparation
4. Model Training
5. Inference and Testing

In [ ]:
# Install dependencies (run this in Colab)
!pip install -qU transformers datasets optimum
!pip install -qU openai wandb
!pip install -qU json-repair pydantic pyyaml
!pip install -qU unsloth bitsandbytes accelerate peft trl

## 1. Setup and Configuration

In [ ]:
# Import required libraries
import sys
import os
from pathlib import Path

# For Colab: Mount Google Drive
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    IN_COLAB = True
    
    # Get credentials from Colab secrets
    import wandb
    wandb.login(key=userdata.get('wandb'))
    hf_token = userdata.get('huggingface')
    openrouter_key = userdata.get('openrouter')
    !huggingface-cli login --token {hf_token}
    
    # Set project directory
    project_dir = "/content/drive/MyDrive/finetuning"
    
except ImportError:
    IN_COLAB = False
    # For local: Load from api_credentials.py
    project_dir = Path(__file__).parent.parent if '__file__' in globals() else Path.cwd().parent
    sys.path.append(str(project_dir))
    
    from api_credentials import HUGGINGFACE_TOKEN, OPENROUTER_API_KEY, WANDB_API_KEY
    hf_token = HUGGINGFACE_TOKEN
    openrouter_key = OPENROUTER_API_KEY
    
    import wandb
    wandb.login(key=WANDB_API_KEY)

# Add src to path
sys.path.append(str(Path(project_dir) / "src"))

print(f"✅ Project directory: {project_dir}")
print(f"✅ Running in {'Colab' if IN_COLAB else 'Local'} environment")

In [ ]:
# Import project modules
from src.data_generator import DataGenerator
from src.data_loader import DataLoader
from src.trainer import ModelTrainer
from src.inference import ModelInference

import json
import random

print("✅ All modules imported successfully")

## 2. Load Raw Data

In [ ]:
# Load raw news data
raw_data_path = Path(project_dir) / "data" / "raw" / "news-sample.jsonl"

raw_data = DataLoader.load_raw_data(str(raw_data_path))
random.Random(42).shuffle(raw_data)

print(f"✅ Loaded {len(raw_data)} raw stories")
print(f"\nSample story preview:")
print(raw_data[0]['content'][:200] + "...")

## 3. Generate SFT Data (Knowledge Distillation)

This step uses DeepSeek API via OpenRouter to generate high-quality training data.

In [ ]:
# Initialize data generator
generator = DataGenerator(
    api_key=openrouter_key,
    model_id="deepseek/deepseek-v3.1-terminus",
    temperature=0.2,
    max_tokens=512
)

# Generate SFT dataset
sft_output_path = Path(project_dir) / "data" / "processed" / "sft.jsonl"

# Choose how many samples to process
max_samples = 50  # Set to None to process all data

generator.generate_sft_dataset(
    raw_data=raw_data,
    output_path=str(sft_output_path),
    include_translations=True,
    target_languages=["English", "French"],
    max_samples=max_samples
)

## 4. Prepare Training Dataset

In [ ]:
# Load and prepare dataset
train_dataset, eval_dataset = DataLoader.load_and_prepare(
    sft_data_path=str(sft_output_path),
    test_size=0.1,
    seed=42
)

print(f"\n✅ Dataset prepared:")
print(f"   Training examples: {len(train_dataset)}")
print(f"   Evaluation examples: {len(eval_dataset)}")

# Preview training example
print(f"\nTraining example preview:")
example = train_dataset[0]
for msg in example["messages"]:
    print(f"\n{msg['role'].upper()}:")
    content = msg['content']
    print(content[:150] + "..." if len(content) > 150 else content)

## 5. Train Model

In [ ]:
# Load configuration
config_path = Path(project_dir) / "config" / "training_config.yaml"

# Initialize trainer
trainer = ModelTrainer(config_path=str(config_path))

# Load model with LoRA
trainer.load_model(hf_token=hf_token)

In [ ]:
# Prepare trainer
trainer.prepare_trainer(
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

In [ ]:
# Start training
trainer.train()

## 6. Save Model

In [ ]:
# Save trained model
model_output_path = Path(project_dir) / "outputs" / "models" / "qwen-arabic-assistant-final"

trainer.save_model(str(model_output_path))

print(f"✅ Model saved to: {model_output_path}")

## 7. Test Model (Inference)

In [ ]:
# Load model for inference
inference = ModelInference(
    model_path=str(model_output_path),
    max_seq_length=2048
)

print("✅ Model loaded for inference")

In [ ]:
# Test with sample data
sft_data = DataLoader.load_sft_data(str(sft_output_path), filter_successful=False)
test_sample = sft_data[0]

story = test_sample["story"]
task = test_sample["task"]

print("📰 Testing finetuned model:\n")
print(f"Story: {story[:150]}...\n")
print(f"Task: {task}\n")

response = inference.generate(story, task)

print(f"🤖 Model Response:")
print(response)

In [ ]:
# Test extraction
test_story = raw_data[5]['content']

print("📰 Test Story:")
print(test_story[:200] + "...\n")

details = inference.extract_details(test_story)

print("🤖 Extracted Details:")
print(details)

In [ ]:
# Test translation
translation = inference.translate(test_story, "English")

print("🤖 Translation:")
print(translation)

## 8. Optional: Push to Hugging Face Hub

In [ ]:
# Uncomment to push model to Hugging Face Hub
# hf_username = "YOUR_USERNAME"
# model_name = "qwen-arabic-assistant"

# trainer.model.push_to_hub(f"{hf_username}/{model_name}", token=hf_token)
# trainer.tokenizer.push_to_hub(f"{hf_username}/{model_name}", token=hf_token)

# print(f"✅ Model pushed to: https://huggingface.co/{hf_username}/{model_name}")